<h4>The Final Product with GUI</h4>

In [1]:
# !pip install scikeras
# !pip install backtrader
# !pip install keras
# !pip install tensorflow
# !pip install yfinance
# !pip install tensorflow-cpu
# !pip install bbc-feeds

In [2]:
# Libraries for machine learning
import sklearn
from sklearn.linear_model import LinearRegression
import keras
from keras import layers, models, optimizers
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import SGD
from keras.layers import LSTM
from scikeras.wrappers import KerasRegressor
from keras import backend as K
from keras.models import load_model
from keras.optimizers import Adam
from IPython.core.debugger import set_trace
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

# Libraries for Statistical Models
import statsmodels.api as sm

# Libraries for Saving the Model
from pickle import dump
from pickle import load

# Libraries for Time series Models
from statsmodels.tsa.arima.model import ARIMA

# Libraries for Error Metrics
from sklearn.metrics import mean_squared_error

# Libraries for Feature Selection
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2, f_regression

# Libraries for Plotting
from pandas.plotting import scatter_matrix
from statsmodels.graphics.tsaplots import plot_acf

# Import for trade simulation
import backtrader as bt
import backtrader.indicators as btind
import backtrader.analyzers as btanalyzers

# Import other general use libraries
import numpy as np
from numpy.random import choice
import pandas as pd
from pandas import read_csv, set_option
from pandas.plotting import scatter_matrix
import yfinance as yf
import matplotlib.pyplot as plt
from matplotlib import pyplot
from pandas.plotting import scatter_matrix
import datetime
from datetime import date, datetime, timedelta
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import joblib
import math
import random
from collections import namedtuple, deque
from collections import deque
import statistics
import bbc_feeds
import tkinter as tk

In [9]:
def runModel():
    # the user input
    target_tickers = [user_input.get()]
    
    # ============Technical Analysis (start)============
    
    reference_tickers = ['^GSPC', '^IXIC', '^DJI', '^VIX', 'DX-Y.NYB', 'BTC-USD']
    
    # get all the features needed for training the model and estimating the return
    endDate = date.today()
    startDate = endDate - timedelta(days = 365)
    target_data = yf.download(target_tickers, start = startDate, end = endDate, interval = '1d')
    reference_data = yf.download(reference_tickers, start = startDate, end = endDate, interval = '1d')
    
    # building own indicators
    df = target_data
    df = df.dropna()
    df = df.sort_values(by = 'Date', ascending = True)
    df.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
    df['prevReturn_1Ddelay'] = df['Open'].pct_change()
    df['curReturn'] = df['prevReturn_1Ddelay'].shift(-1)
    df['prevReturn_2Ddelay'] = df['curReturn'].shift(2)
    df['prevReturn_3Ddelay'] = df['curReturn'].shift(3)
    df['prevReturn_4Ddelay'] = df['curReturn'].shift(4)
    df['prevReturn_5Ddelay'] = df['curReturn'].shift(5)
    
    # rate of change
    def ROC(df, n):
      M = df.diff(n - 1)
      N = df.shift(n - 1)
      ROC = pd.Series(((M / N) * 100), name = 'ROC_' + str(n))
      return ROC
    df['ROC_10'] = ROC(df['Open'], 10)
    df['ROC_20'] = ROC(df['Open'], 20)
    df['ROC_30'] = ROC(df['Open'], 30)
    
    # price momentum
    def MOM(df, n):
      MOM = pd.Series(df.diff(n), name = 'Momentum_' + str(n))
      return MOM
    df['MOM_10'] = MOM(df['Open'], 10)
    df['MOM_20'] = MOM(df['Open'], 20)
    df['MOM_30'] = MOM(df['Open'], 30)
    
    # relative strength index
    def RSI(series, period):
      delta = series.diff().dropna()
      u = delta * 0
      d = u.copy()
      u[delta > 0] = delta[delta > 0]
      d[delta < 0] = -delta[delta < 0]
      u[u.index[period-1]] = np.mean( u[:period] )
      u = u.drop(u.index[:(period - 1)])
      d[d.index[period - 1]] = np.mean( d[:period] )
      d = d.drop(d.index[:(period - 1)])
      rs = u.ewm(com = period - 1, adjust = False).mean() / d.ewm(com = period - 1, adjust = False).mean()
      return 100 - 100 / (1 + rs)
    df['RSI_15'] = RSI(df['Open'], 15)
    df['RSI_30'] = RSI(df['Open'], 30)
    df['RSI_60'] = RSI(df['Open'], 60)
    
    # stochastic oscillator
    def STOK(close, low, high, n):
      STOK = ((close - low.rolling(n).min()) / (high.rolling(n).max() - low.rolling(n).min())) * 100
      return STOK
    def STOD(close, low, high, n):
      STOK = ((close - low.rolling(n).min()) / (high.rolling(n).max() - low.rolling(n).min())) * 100
      STOD = STOK.rolling(3).mean()
      return STOD
    df['STOD_%K_15'] = STOK(df['Close'], df['Low'], df['High'], 15)
    df['STOD_%K_30'] = STOK(df['Close'], df['Low'], df['High'], 30)
    df['STOD_%K_60'] = STOK(df['Close'], df['Low'], df['High'], 60)
    df['STOD_%D_15'] = STOD(df['Close'], df['Low'], df['High'], 15)
    df['STOD_%D_30'] = STOD(df['Close'], df['Low'], df['High'], 30)
    df['STOD_%D_60'] = STOD(df['Close'], df['Low'], df['High'], 60)
    
    df = df[['curReturn', 'prevReturn_1Ddelay', 'prevReturn_2Ddelay', 'prevReturn_3Ddelay', 'prevReturn_4Ddelay', 'prevReturn_5Ddelay', 'ROC_10', 'ROC_20', 'ROC_30', 'MOM_10', 'MOM_20', 'MOM_30', 'RSI_15', 'RSI_30', 'RSI_60', 'STOD_%K_15', 'STOD_%K_30', 'STOD_%K_60', 'STOD_%D_15', 'STOD_%D_30', 'STOD_%D_60']]
    
    # other financial asset for reference - previous day return only
    df_other = reference_data['Open']
    df_other = df_other.dropna()
    df_other = df_other.sort_values(by = 'Date', ascending = True)
    df_other = df_other.pct_change()
    df = pd.merge(df, df_other, how = 'left', left_on = ['Date'], right_on = ['Date'])
    df = df.dropna()
    df = df.sort_values(by = 'Date', ascending = True)
    df = df.loc[:, df.columns != target_tickers[0]]
    
    # Ready Data for training
    Y = df.loc[:, 'curReturn']
    Y = Y.iloc[:-1]
    X = df.loc[:, df.columns != 'curReturn']
    X = X.iloc[:-1]
    
    # start training the linear regression model
    model = LinearRegression()
    model.fit(X, Y)
    
    # calculate the latest estimated return of the coming day
    x = df.tail(n = 1)
    x = x.loc[:, x.columns != 'curReturn']
    TA_indicatorValue = model.predict(x)
    TA_indicatorValue = float(TA_indicatorValue)
    
    # ============Technical Analysis (end)============
    
    # ============Reinforcement Learning (start)============
    
    # get latest finance data
    window_size = 10
    endDate = date.today()
    startDate = endDate - timedelta(days = window_size * 2) # just to ensure enough data if some days is not trading day
    financeData = yf.download(target_tickers, start = startDate, end = endDate, interval = '1d')
    financeData.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
    financeData = financeData.dropna()
    financeData = financeData.sort_values(by = 'Date', ascending = True)
    financeData['prevReturn'] = financeData['Open'].pct_change()
    financeData = financeData.dropna()
    financeData = financeData.tail(window_size)
    
    # load the RL model previously saved
    model_RL_loaded = load_model('RL_model/RL_model_final.keras')
    X = list(financeData["prevReturn"])
    X = np.array([[float(x) for x in X]])
    RL_indicatorValue = np.argmax(model_RL_loaded.predict(X)[0])
    
    # ============Reinforcement Learning (end)============
    
    # ============Sentiment Analysis (start)============
    
    # get latest news
    news_stories = bbc_feeds.news().all()
    titles = []
    for story in news_stories:
      titles.append(story['title'])
    
    # load the LSTM model previously saved
    model_LSTM_loaded = keras.models.load_model('SA_model/SA_model.keras')
    tokenizer = Tokenizer(num_words = 40000)
    sequences_LSTM = tokenizer.texts_to_sequences(titles)
    X_LSTM = pad_sequences(sequences_LSTM, maxlen = 100)
    Y_LSTM = model_LSTM_loaded.predict(X_LSTM)
    SA_indicatorValue = Y_LSTM.mean()
    
    # ============Sentiment Analysis (end)============
    
    # ============Combined Model (start)============
    
    CombinedModel_loaded = joblib.load('Combined_model.sav')
    x = pd.DataFrame([[TA_indicatorValue, RL_indicatorValue, SA_indicatorValue]], columns = ['TA_estReturn', 'RL_signal', 'SA_sentiment'])
    # x = [TA_indicatorValue, RL_indicatorValue, SA_indicatorValue]
    suggestion = CombinedModel_loaded.predict(x)
    suggestion = suggestion[0]
    
    # ============Combined Model (end)============

    # ============GUI control (start)============
    
    TA_res.config(text = round(TA_indicatorValue, 6))
    RL_res.config(text = int(RL_indicatorValue))
    SA_res.config(text = round(SA_indicatorValue, 6))
    suggestion = 'BUY (or HOLD if in position already)' if int(suggestion) == 1 else 'SELL (or HOLD if not in position)'
    combined_res.config(text = suggestion)

    # ============GUI control (end)============

    return round(TA_indicatorValue, 6), int(RL_indicatorValue), round(SA_indicatorValue, 6), int(suggestion)

In [12]:
root = tk.Tk()
root.title('Daily Investment Advisor')
root.geometry('400x250')

label_ticker = tk.Label(root, text = 'Please enter the ticker:').pack()
user_input = tk.Entry(root)
user_input.pack()
generate_button = tk.Button(root, text = 'Generate Suggestion', command = runModel).pack()

TA_label = tk.Label(root, text = 'TA_estReturn').place(x = 60, y = 100, anchor = tk.CENTER)
RL_label = tk.Label(root, text = 'RL_signal').place(x = 200, y = 100, anchor = tk.CENTER)
SA_label = tk.Label(root, text = 'SA_sentiment').place(x = 340, y = 100, anchor = tk.CENTER)
TA_res = tk.Label(root, text = '--------', fg = 'blue')
TA_res.place(x = 60, y = 125, anchor = tk.CENTER)
RL_res = tk.Label(root, text = '--------', fg = 'blue')
RL_res.place(x = 200, y = 125, anchor = tk.CENTER)
SA_res = tk.Label(root, text = '--------', fg = 'blue')
SA_res.place(x = 340, y = 125, anchor = tk.CENTER)

combined_label = tk.Label(root, text = 'Combined Suggestion').place(x = 200, y = 180, anchor = tk.CENTER)
combined_res = tk.Label(root, text = '--------', fg = 'blue')
combined_res.place(x = 200, y = 205, anchor = tk.CENTER)

root.mainloop()

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 123"}}}
$123: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['123']: possibly delisted; no timezone found
[*********************100%***********************]  6 of 6 completed
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\tkinter\__init__.py", line 2074, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\ziv_s\AppData\Local\Temp\ipykernel_24968\129851188.py", line 19, in runModel
    df.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
    ^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\site-packages\pandas\core\generic.py", line 6335, in __setattr__
    return object.__setattr__(self, name, value)
           ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "pandas/_libs/properties.pyx", line 69, in panda